In [ ]:
import sys
from pathlib import Path

current_dir = Path.cwd().resolve()

# Find 'src' by walking up the directory tree
src_path = None
for parent in [current_dir] + list(current_dir.parents):
    candidate = parent / 'src'
    if candidate.exists() and candidate.is_dir():
        src_path = candidate
        break

if src_path:
    if str(src_path) not in sys.path:
        sys.path.append(str(src_path))
    print(f"Success: Linked to src at {src_path}")
    
    # Import
    try:
        from config import *
        print(f"Environment linked. Data dir: {DATA_DIR}")    
    except ImportError as e:
        print(f"CRITICAL: Found src but could not import config. {e}")
else:
    print("CRITICAL ERROR: Could not find 'src' directory in any parent folder.")
    print(f"Current search path: {current_dir}")

In [ ]:
import numpy as np
from scipy import signal
import os

OUTPUT_DIR = UNET_PROCESSED_DATA_DIR
OUTPUT_FILENAME = f"{BarrageSignal}_raw_data_clean.npy"  # skips the dc offset removal hence the "clean" in the name, but it's really just the raw data
N_FILES = 1000                # Number of unique noise "files" to generate
SAMPLE_LENGTH = 40960         # Length of one signal frame
SEED = 42                     # Ensure reproducibility

print(f"Configuration Set: Generating {N_FILES} files of length {SAMPLE_LENGTH}...")

In [ ]:
def generate_awgn(n_files, length, seed=None):
    """
    Generates Complex Gaussian White Noise (AWGN) with Unity Power.
    
    Physics:
    - To achieve E[|z|^2] = 1 (Unity Power), the variance of Real and Imag
      components must each be 0.5.
      Var(z) = Var(x) + Var(y) = 0.5 + 0.5 = 1.0
    """
    if seed:
        np.random.seed(seed)
    
    # Standard Deviation calculation
    # sigma = sqrt(variance) = sqrt(0.5) approx 0.707
    sigma = np.sqrt(0.5)
    
    print(f"Generating noise with Sigma={sigma:.4f} per component...")
    
    # Generate Real and Imaginary parts independently
    # Shape: (N_FILES, LENGTH)
    noise_i = np.random.normal(0, sigma, size=(n_files, length)).astype(np.float32)
    noise_q = np.random.normal(0, sigma, size=(n_files, length)).astype(np.float32)
    
    # Stack into (N, 2, L) format to match dataset standard
    # Axis 1 = [I, Q]
    batch_tensor = np.stack([noise_i, noise_q], axis=1)
    
    return batch_tensor

In [ ]:
# --- Execution ---
print("Starting Generation...")
barrage_tensor = generate_awgn(N_FILES, SAMPLE_LENGTH, SEED)

print(f"Generation Complete.")
print(f"Output Shape: {barrage_tensor.shape} (Files, Channels, Length)")

# --- Save to Disk ---
save_path = os.path.join(OUTPUT_DIR, OUTPUT_FILENAME)
np.save(save_path, barrage_tensor)
print(f"\nFile saved successfully to: {save_path}")
print(f"File Size: {os.path.getsize(save_path) / (1024**3):.2f} GB")